<a href="https://colab.research.google.com/github/Ashleylq/house-price-regressor/blob/main/notebooks/Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn numpy

# **Load Dataset**

In [ ]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/wblakecannon/ames/refs/heads/master/data/housing.csv")

# **Train Test Split**

In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# **Get Numerical and Categorical Features**

In [ ]:
num = X_train.select_dtypes(include=["number"]).columns.tolist()
cat = X_train.select_dtypes(exclude=["number"]).columns.tolist()
num.remove("MS SubClass")
cat.append("MS SubClass")

# **Create Preprocessing Function**

In [19]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="constant", fill_value="missing")),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

std_preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num),
    ('cat', categorical_transformer, cat)
])

We have made a transformer with only scaling and imputing so lets make another one with log transform aswell

In [ ]:
from sklearn.preprocessing import FunctionTransformer
import numpy as np

skewed_columns = [
 'Lot Frontage',
 'Lot Area',
 'Mas Vnr Area',
 'BsmtFin SF 1',
 'BsmtFin SF 2',
 'Total Bsmt SF',
 '1st Flr SF',
 'Low Qual Fin SF',
 'Gr Liv Area',
 'Bsmt Half Bath',
 'Kitchen AbvGr',
 'Wood Deck SF',
 'Open Porch SF',
 'Enclosed Porch',
 '3Ssn Porch',
 'Screen Porch',
 'Pool Area',
 'Misc Val']

not_skewed = [col for col in num if col not in skewed_columns]

log_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('log1p', FunctionTransformer(func=np.log1p)),
    ('scaler', StandardScaler())
])

log_preprocessor = ColumnTransformer(transformers=[
    ('log', log_transformer, skewed_columns),
    ('num', numeric_transformer, not_skewed),
    ('cat', categorical_transformer, cat)
])

# **Compare Preprocessing Methods**

In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

experiments = [
    {
        "name" : "stdEN",
        "model" : ElasticNet,
        "preprocessor" : std_preprocessor
    },{
        "name" : "logEN",
        "model" : ElasticNet,
        "preprocessor" : log_preprocessor
    },{
        "name" : "stdRF",
        "model" : RandomForestRegressor,
        "preprocessor" : std_preprocessor
    },{
        "name" : "logRF",
        "model" : RandomForestRegressor,
        "preprocessor" : log_preprocessor
    },{
        "name" : "stdGB",
        "model" : GradientBoostingRegressor,
        "preprocessor" : std_preprocessor,
    },{
        "name" : "logGB",
        "model" : GradientBoostingRegressor,
        "preprocessor" : log_preprocessor
    }
]

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import TransformedTargetRegressor

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def conduct_experiments(experiments):
  results = []
  for i, exp in enumerate(experiments):
    model = TransformedTargetRegressor(
           regressor=exp["model"](),
           func=np.log1p,
           inverse_func=np.expm1
        )
    pipeline = Pipeline([
        ("preprocessor", exp["preprocessor"]),
        ("model", model)
    ])
    score = cross_val_score(pipeline, X, y, cv=cv, scoring="neg_root_mean_squared_error")
    results.append({"name":exp["name"], "score":-score.mean()})
  print(results)

In [ ]:
conduct_experiments(experiments)

stdEN
81001.77110174487
logEN
81001.77110174487
stdRF
26616.018653826362
logRF
26502.989615816372
stdGB
23192.838512318405
logGB
23381.05909825464
[{'name': 'stdEN', 'score': np.float64(81001.77110174487)}, {'name': 'logEN', 'score': np.float64(81001.77110174487)}, {'name': 'stdRF', 'score': np.float64(26616.018653826362)}, {'name': 'logRF', 'score': np.float64(26502.989615816372)}, {'name': 'stdGB', 'score': np.float64(23192.838512318405)}, {'name': 'logGB', 'score': np.float64(23381.05909825464)}]


The log preprocessor did better for all of the models except elastic net where it turned out to have the same loss as the standard preprocessor

# **Feature Engineering**

In [21]:
def apply_log(X):
  copy = X.copy()
  copy[skewed_columns] = copy[skewed_columns].apply(np.log1p)
  return copy

def create_features(X):
  copy = X.copy()
  copy['Tot Liv Area'] = copy['Total Bsmt SF'] + copy['Gr Liv Area']
  copy['Tot Rooms'] = copy['TotRms AbvGrd'] + copy['Bsmt Half Bath'] + copy['Bsmt Full Bath']
  copy['HouseAge'] = copy['Yr Sold'] - copy['Year Built']
  copy['Yrs Since Remod'] = copy['Yr Sold'] - copy['Year Remod/Add']
  return copy

eng_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('engineer', FunctionTransformer(func=create_features)),
    ('log1p', FunctionTransformer(func=apply_log)),
    ('scaler', StandardScaler())
])

eng_preprocessor = ColumnTransformer(transformers=[
    ('num', eng_transformer, num),
    ('cat', categorical_transformer, cat)
])

eng_preprocessor.set_output(transform="pandas")

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('engineer',
                                                  FunctionTransformer(func=<function create_features at 0x7c2325f1f9c0>)),
                                                 ('log1p',
                                                  FunctionTransformer(func=<function apply_log at 0x7c2325f1c0e0>)),
                                                 ('scaler', StandardScaler())]),
                                 ['Unnamed: 0', 'Order', 'PID', 'Lot Frontage',
                                  'Lot Area', 'Overall Q...
                                 ['MS Zoning', 'Street', 'Alley', 'Lot Shape',
                                  'Land Contour', 'Utilities', 'Lot Config',
                                  'Land Slope', 'Neighborhood', 'Condition 1',
                                  'Condition 2', 'Bldg Type', 'House Style',
                                  'Roof Style', 'Roof Matl', 'Exterior 1st',
                                  'Exterior 2nd', 'Mas Vnr Type', 'Exter Qual',
                                  'Exter Cond', 'Foundation', 'Bsmt Qual',
                                  'Bsmt Cond', 'Bsmt Exposure',
                                  'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating',
                                  'Heating QC', 'Central Air', 'Electrical', ...])])

In [22]:
experiments = [{
    "name" : "engEN",
    "model" : ElasticNet,
    "preprocessor" : eng_preprocessor
}, {
    "name" : "engRF",
    "model" : RandomForestRegressor,
    "preprocessor" : eng_preprocessor
}, {
    "name" : "engGB",
    "model" : GradientBoostingRegressor,
    "preprocessor" : eng_preprocessor
}]

conduct_experiments(experiments)

engEN
81001.77110174487
engRF
25391.747614776992
engGB
22778.20150666266
[{'name': 'engEN', 'score': np.float64(81001.77110174487)}, {'name': 'engRF', 'score': np.float64(25391.747614776992)}, {'name': 'engGB', 'score': np.float64(22778.20150666266)}]


Gradient Boosting Regressor with engineered features and log1p is the best performing model.

# **Hyperparameter Tuning with Randomized Search CV**

In [26]:
pipeline = Pipeline(steps=[
    ("preprocessor", eng_preprocessor),
    ("model", TransformedTargetRegressor(
        regressor=GradientBoostingRegressor(),
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

pipeline.get_params().keys()

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'preprocessor', 'model', 'preprocessor__force_int_remainder_cols', 'preprocessor__n_jobs', 'preprocessor__remainder', 'preprocessor__sparse_threshold', 'preprocessor__transformer_weights', 'preprocessor__transformers', 'preprocessor__verbose', 'preprocessor__verbose_feature_names_out', 'preprocessor__num', 'preprocessor__cat', 'preprocessor__num__memory', 'preprocessor__num__steps', 'preprocessor__num__transform_input', 'preprocessor__num__verbose', 'preprocessor__num__imputer', 'preprocessor__num__engineer', 'preprocessor__num__log1p', 'preprocessor__num__scaler', 'preprocessor__num__imputer__add_indicator', 'preprocessor__num__imputer__copy', 'preprocessor__num__imputer__fill_value', 'preprocessor__num__imputer__keep_empty_features', 'preprocessor__num__imputer__missing_values', 'preprocessor__num__imputer__strategy', 'preprocessor__num__engineer__accept_sparse', 'preprocessor__num__engineer__check_inverse', 'preprocessor__n

In [27]:
param_distributions = {
    "model__regressor__learning_rate" : [0.01, 0.03, 0.05, 0.1],
    "model__regressor__n_estimators" : [100, 200, 300, 400],
    "model__regressor__max_depth" : [3, 4, 5],
    "model__regressor__subsample" : [0.8, 1],
    "model__regressor__min_samples_leaf" : [1, 2, 4],
    "model__regressor__min_samples_split" : [2, 5, 10]
}

In [30]:
from sklearn.model_selection import RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=42)
search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='most_frequent')),
                                                                                               ('engineer',
                                                                                                FunctionTransformer(func=<function create_features at 0x7c2325f1f9c0>)),
                                                                                               ('log1p',
                                                                                                FunctionTransformer(func=<function apply_log at 0x7c2325f1c0e0>)),
                                                                                               ('scaler',
                                                                                                StandardScaler...
                   n_iter=50, n_jobs=-1,
                   param_distributions={'model__regressor__learning_rate': [0.01,
                                                                            0.03,
                                                                            0.05,
                                                                            0.1],
                                        'model__regressor__max_depth': [3, 4,
                                                                        5],
                                        'model__regressor__min_samples_leaf': [1,
                                                                               2,
                                                                               4],
                                        'model__regressor__min_samples_split': [2,
                                                                                5,
                                                                                10],
                                        'model__regressor__n_estimators': [100,
                                                                           200,
                                                                           300,
                                                                           400],
                                        'model__regressor__subsample': [0.8,
                                                                        1]},
                   random_state=42, scoring='neg_mean_squared_error')

In [31]:
search.best_params_

{'model__regressor__subsample': 0.8,
 'model__regressor__n_estimators': 200,
 'model__regressor__min_samples_split': 10,
 'model__regressor__min_samples_leaf': 1,
 'model__regressor__max_depth': 4,
 'model__regressor__learning_rate': 0.05}

So the best hyperparameters are :

*   learning rate : 0.05
*   max depth : 4
*   n estimators : 200
*   min samples split : 10
*   min samples leaf : 1
*   subsample : 0.8

In [32]:
from sklearn.metrics import mean_squared_error, r2_score
model = search.best_estimator_
pred = model.predict(X_test)
mse = mean_squared_error(y_test, pred)
r2 = r2_score(y_test, pred)
mse, r2

(566887463.6745398, 0.9292941596907478)

The model made from these hyperparameters scored an R2 score of 0.93 on our test set.